In [2]:
!pip install -q transformers datasets

import torch
from datasets import load_dataset

print("GPU available:", torch.cuda.is_available())

# Free dataset: 560k business reviews labelled positive (1) / negative (0)
data = load_dataset("fancyzhx/yelp_polarity")

# Use a small sample so training is fast
train = data["train"].shuffle(seed=42).select(range(4000))
test  = data["test"].shuffle(seed=42).select(range(1000))

print(train)
print(train[0])

GPU available: True


README.md:   0%|          | 0.00/8.93k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  256MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 17.7MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/38000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label'],
    num_rows: 4000
})
{'text': "Decent size, decent selection, decent staff.\\n\\nI guess that can wholly sum this place up, it's decent.  As with many other stores that are like this, the product rotates depending on what doesn't sale well at other stores.  Can always snag a deal here.  I was able to pick up a pretty sweet Puma jacket for $10, can't beat that, right?\\n\\nThat being said, there are those times that you may not find anything as well.  So really don't get your hopes up if you are looking for a specific item.", 'label': 1}


In [8]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReviewDataset(Dataset):
    def __init__(self, hf_data):
        texts = list(hf_data["text"])
        labels = list(hf_data["label"])
        enc = tokenizer(texts, truncation=True, padding="max_length", max_length=128)
        self.input_ids = torch.tensor(enc["input_ids"])
        self.attention_mask = torch.tensor(enc["attention_mask"])
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {"input_ids": self.input_ids[idx],
                "attention_mask": self.attention_mask[idx],
                "labels": self.labels[idx]}

train_ds = ReviewDataset(train)
test_ds = ReviewDataset(test)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32)

batch = next(iter(train_loader))
print({k: v.shape for k, v in batch.items()})

{'input_ids': torch.Size([16, 128]), 'attention_mask': torch.Size([16, 128]), 'labels': torch.Size([16])}


In [9]:
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Pre-trained DistilBERT with a new 2-class output layer (negative / positive)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
EPOCHS = 2

def evaluate(model, loader):
    model.eval()                      # switch off training behaviour (dropout)
    preds, labels = [], []
    with torch.no_grad():             # no gradients needed for evaluation
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(input_ids=batch["input_ids"],
                           attention_mask=batch["attention_mask"]).logits
            preds += logits.argmax(dim=1).cpu().tolist()
            labels += batch["labels"].cpu().tolist()
    return accuracy_score(labels, preds), f1_score(labels, preds)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)      # forward pass (loss computed from labels)
        loss = outputs.loss

        optimizer.zero_grad()         # reset old gradients
        loss.backward()               # backpropagation
        optimizer.step()              # update the weights

        total_loss += loss.item()
        if step % 50 == 0:
            print(f"Epoch {epoch+1} | step {step} | loss {loss.item():.4f}")

    acc, f1 = evaluate(model, test_loader)
    print(f"Epoch {epoch+1} done | avg loss {total_loss/len(train_loader):.4f} "
          f"| test accuracy {acc:.3f} | F1 {f1:.3f}")

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1 | step 0 | loss 0.6757
Epoch 1 | step 50 | loss 0.2201
Epoch 1 | step 100 | loss 0.1566
Epoch 1 | step 150 | loss 0.2733
Epoch 1 | step 200 | loss 0.5621
Epoch 1 done | avg loss 0.3540 | test accuracy 0.873 | F1 0.880
Epoch 2 | step 0 | loss 0.1321
Epoch 2 | step 50 | loss 0.4158
Epoch 2 | step 100 | loss 0.1807
Epoch 2 | step 150 | loss 0.1214
Epoch 2 | step 200 | loss 0.3556
Epoch 2 done | avg loss 0.1827 | test accuracy 0.913 | F1 0.905


In [10]:
!pip install -q fastapi httpx

# 1. Save the fine-tuned model and tokenizer
model.save_pretrained("review_model")
tokenizer.save_pretrained("review_model")

# 2. Write the FastAPI app to a file
app_code = '''
import torch
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = "review_model"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()

app = FastAPI(title="Review Sentiment API")

class Review(BaseModel):
    text: str

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(review: Review):
    enc = tokenizer(review.text, truncation=True, max_length=128, return_tensors="pt")
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits, dim=1)[0]
    label = int(probs.argmax())
    return {"sentiment": "positive" if label == 1 else "negative",
            "confidence": round(float(probs[label]), 3)}
'''
with open("app.py", "w") as f:
    f.write(app_code)

# 3. Test the API without a server, using FastAPI's TestClient
from fastapi.testclient import TestClient
import importlib, app as review_app
importlib.reload(review_app)
client = TestClient(review_app.app)

print(client.get("/health").json())
print(client.post("/predict", json={"text": "The room was spotless and the staff were lovely!"}).json())
print(client.post("/predict", json={"text": "Dirty bathroom, rude reception, never again."}).json())

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'status': 'ok'}
{'sentiment': 'positive', 'confidence': 0.941}
{'sentiment': 'negative', 'confidence': 0.992}


In [11]:
readme = """# Review Sentiment Classifier - PyTorch + FastAPI

Fine-tuned **DistilBERT** for binary sentiment classification of customer reviews,
with a custom **PyTorch** training loop and a **FastAPI** inference service.

## Results
- Training data: 4,000 Yelp reviews | Test data: 1,000 unseen reviews
- **Test accuracy: 91.3% | F1: 0.905** after 2 epochs

## How it works
1. Tokenisation with a custom PyTorch `Dataset` and `DataLoader`
2. Fine-tuning DistilBERT with a manual training loop (forward pass, loss, backpropagation, AdamW)
3. Evaluation with accuracy and F1 on a held-out test set
4. Serving the model via FastAPI (`/health`, `/predict`), containerised with Docker

## Run the API
    pip install -r requirements.txt
    uvicorn app:app --reload
    # POST /predict  {"text": "Great hotel, friendly staff!"}

Model weights are not stored in this repo (size); train them with the notebook.
"""
reqs = "torch\ntransformers\nfastapi\nuvicorn\npydantic\n"
docker = """FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""
for name, content in [("README.md", readme), ("requirements.txt", reqs), ("Dockerfile", docker)]:
    with open(name, "w") as f:
        f.write(content)

!zip -q project_files.zip README.md requirements.txt Dockerfile app.py
from google.colab import files
files.download("project_files.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>